# 🔧 Feature Engineering - AI LogGuard Phase 3

**Mục đích:** Trích xuất features từ CI/CD logs để training ML model

**Features:**
1. **Text Features:** TF-IDF vectorization
2. **Structural Features:** log length, error patterns, keywords
3. **Platform Features:** one-hot encoding

## 1. Setup

In [ ]:
!pip install pandas scikit-learn nltk joblib -q

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries imported!")

## 2. Load Data

In [ ]:
# Load splits từ notebook 01
train_df = pd.read_csv('data/synthetic_logs/train.csv')
val_df = pd.read_csv('data/synthetic_logs/val.csv')
test_df = pd.read_csv('data/synthetic_logs/test.csv')

print(f"Train: {len(train_df)} samples")
print(f"Val:   {len(val_df)} samples")
print(f"Test:  {len(test_df)} samples")

## 3. Load Log Contents

In [ ]:
def load_log_content(file_path):
    """Load content from log file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return ""

# Load log contents
print("📂 Loading log contents...")
train_df['log_content'] = train_df['file_path'].apply(load_log_content)
val_df['log_content'] = val_df['file_path'].apply(load_log_content)
test_df['log_content'] = test_df['file_path'].apply(load_log_content)

print("✅ Log contents loaded!")
print(f"\nSample log (first 500 chars):")
print(train_df['log_content'].iloc[0][:500])

## 4. Text Preprocessing

In [ ]:
def preprocess_text(text):
    """Clean and preprocess log text"""
    if not text:
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove timestamps (common patterns)
    text = re.sub(r'\d{4}-\d{2}-\d{2}[T\s]\d{2}:\d{2}:\d{2}', '', text)
    text = re.sub(r'\d{2}:\d{2}:\d{2}', '', text)
    
    # Remove URLs
    text = re.sub(r'https?://\S+', 'URL', text)
    
    # Remove file paths (but keep file names)
    text = re.sub(r'/[a-z0-9_\-/]+/', ' ', text)
    
    # Remove build numbers
    text = re.sub(r'#\d+', '', text)
    
    # Remove version numbers (keep semantic meaning)
    text = re.sub(r'\d+\.\d+\.\d+', 'VERSION', text)
    
    # Keep error codes (important!)
    # E.g., E404, ERESOLVE, TS2322
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Test preprocessing
sample = train_df['log_content'].iloc[0][:200]
print("Before:")
print(sample)
print("\nAfter:")
print(preprocess_text(sample))

In [ ]:
# Apply preprocessing
print("🔧 Preprocessing logs...")
train_df['log_clean'] = train_df['log_content'].apply(preprocess_text)
val_df['log_clean'] = val_df['log_content'].apply(preprocess_text)
test_df['log_clean'] = test_df['log_content'].apply(preprocess_text)
print("✅ Preprocessing complete!")

## 5. TF-IDF Features

In [ ]:
# Create TF-IDF vectorizer
# max_features: số features tối đa (trade-off: accuracy vs speed)
# ngram_range: (1,2) = unigrams + bigrams
# min_df: ignore terms xuất hiện < 2 documents
# max_df: ignore terms xuất hiện > 80% documents

tfidf = TfidfVectorizer(
    max_features=500,  # Giữ 500 features quan trọng nhất
    ngram_range=(1, 2),  # Unigrams + bigrams
    min_df=2,
    max_df=0.8,
    sublinear_tf=True  # Use log scaling
)

# Fit on training data only!
print("🔧 Fitting TF-IDF vectorizer...")
X_train_tfidf = tfidf.fit_transform(train_df['log_clean'])
X_val_tfidf = tfidf.transform(val_df['log_clean'])
X_test_tfidf = tfidf.transform(test_df['log_clean'])

print(f"✅ TF-IDF features created!")
print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

In [ ]:
# Analyze top features
feature_names = tfidf.get_feature_names_out()
print(f"\n🔝 Top 50 TF-IDF features:")
print(feature_names[:50])

## 6. Structural Features

In [ ]:
def extract_structural_features(row):
    """Extract structural features from log"""
    content = row['log_content']
    
    features = {
        # Length features
        'log_length': len(content),
        'num_lines': content.count('\n'),
        
        # Error patterns
        'has_error_keyword': int(bool(re.search(r'\berror\b', content, re.I))),
        'has_failed_keyword': int(bool(re.search(r'\bfailed\b', content, re.I))),
        'has_exception': int(bool(re.search(r'\bexception\b', content, re.I))),
        'has_timeout': int(bool(re.search(r'\btimeout\b|\btimed out\b', content, re.I))),
        
        # Error code patterns
        'has_npm_error': int(bool(re.search(r'\bERR!\b|\bE[A-Z]+\b', content))),
        'has_pip_error': int(bool(re.search(r'\bERROR:\b', content))),
        'has_ts_error': int(bool(re.search(r'\bTS\d+\b', content))),
        'has_syntax_error': int(bool(re.search(r'SyntaxError|IndentationError', content))),
        
        # Test patterns
        'has_test_failed': int(bool(re.search(r'\btest.*failed\b|\bfailed.*test\b', content, re.I))),
        'has_assertion_error': int(bool(re.search(r'AssertionError|expect.*received', content))),
        
        # Stack trace
        'has_stack_trace': int(bool(re.search(r'\s+at\s+.*\(.*:\d+:\d+\)', content))),
        
        # Exit codes
        'has_exit_code': int(bool(re.search(r'exit code|exit status', content, re.I))),
    }
    
    return pd.Series(features)

# Extract structural features
print("🔧 Extracting structural features...")
train_struct = train_df.apply(extract_structural_features, axis=1)
val_struct = val_df.apply(extract_structural_features, axis=1)
test_struct = test_df.apply(extract_structural_features, axis=1)

print("✅ Structural features extracted!")
print(f"\nFeatures: {list(train_struct.columns)}")
print(f"\nSample:")
train_struct.head()

## 7. Platform Features (One-Hot Encoding)

In [ ]:
# One-hot encode platforms
platform_dummies_train = pd.get_dummies(train_df['platform'], prefix='platform')
platform_dummies_val = pd.get_dummies(val_df['platform'], prefix='platform')
platform_dummies_test = pd.get_dummies(test_df['platform'], prefix='platform')

print("Platform features:")
print(platform_dummies_train.columns.tolist())

## 8. Combine All Features

In [ ]:
from scipy.sparse import hstack

# Combine TF-IDF + Structural + Platform features
print("🔧 Combining all features...")

# Convert to arrays
X_train = hstack([
    X_train_tfidf,  # TF-IDF (sparse)
    train_struct.values,  # Structural (dense)
    platform_dummies_train.values  # Platform (dense)
])

X_val = hstack([
    X_val_tfidf,
    val_struct.values,
    platform_dummies_val.values
])

X_test = hstack([
    X_test_tfidf,
    test_struct.values,
    platform_dummies_test.values
])

print(f"✅ Combined features!")
print(f"Train shape: {X_train.shape}")
print(f"Val shape:   {X_val.shape}")
print(f"Test shape:  {X_test.shape}")

## 9. Prepare Labels

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['error_category'])
y_val = label_encoder.transform(val_df['error_category'])
y_test = label_encoder.transform(test_df['error_category'])

print("📊 Label encoding:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {i}: {label}")

print(f"\nLabel distribution in train set:")
unique, counts = np.unique(y_train, return_counts=True)
for label_id, count in zip(unique, counts):
    print(f"  {label_encoder.classes_[label_id]}: {count}")

## 10. Save Features & Preprocessors

In [ ]:
# Create output directory
import os
os.makedirs('models', exist_ok=True)

# Save features
print("💾 Saving features...")
joblib.dump(X_train, 'models/X_train.pkl')
joblib.dump(X_val, 'models/X_val.pkl')
joblib.dump(X_test, 'models/X_test.pkl')
joblib.dump(y_train, 'models/y_train.pkl')
joblib.dump(y_val, 'models/y_val.pkl')
joblib.dump(y_test, 'models/y_test.pkl')

# Save preprocessors (important for inference!)
print("💾 Saving preprocessors...")
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')
joblib.dump(label_encoder, 'models/label_encoder.pkl')

# Save feature info
feature_info = {
    'tfidf_features': X_train_tfidf.shape[1],
    'structural_features': len(train_struct.columns),
    'platform_features': len(platform_dummies_train.columns),
    'total_features': X_train.shape[1],
    'structural_feature_names': train_struct.columns.tolist(),
    'platform_feature_names': platform_dummies_train.columns.tolist()
}
joblib.dump(feature_info, 'models/feature_info.pkl')

print("\n✅ All saved!")
print("Files created:")
print("  - models/X_train.pkl, X_val.pkl, X_test.pkl")
print("  - models/y_train.pkl, y_val.pkl, y_test.pkl")
print("  - models/tfidf_vectorizer.pkl")
print("  - models/label_encoder.pkl")
print("  - models/feature_info.pkl")

## 11. Feature Importance Preview

In [ ]:
# Check structural feature correlations with error categories
print("📊 Structural feature statistics by error category:\n")

for col in train_struct.columns[:5]:  # First 5 features
    print(f"\n{col}:")
    combined = pd.concat([train_df['error_category'], train_struct[col]], axis=1)
    print(combined.groupby('error_category')[col].mean().sort_values(ascending=False))

## ✅ Summary

**Features Created:**
- ✅ TF-IDF features (500 dimensions)
- ✅ Structural features (13 dimensions)
- ✅ Platform features (3 dimensions)
- ✅ Total: ~516 features

**Files Saved:**
- ✅ Feature matrices (X_train, X_val, X_test)
- ✅ Labels (y_train, y_val, y_test)
- ✅ Preprocessors (TF-IDF vectorizer, label encoder)

**Next Step:**
➡️ Notebook 03: Train ML models (Logistic Regression, Random Forest, XGBoost)